In [20]:
symbols = [
    'RELIANCE',
    'HDFCBANK',
    'BHARTIARTL',
    'SBIN',
    'ICICIBANK',
    'TCS',
    'BAJFINANCE',
    'LT',
    'LICI',
    'HINDUNILVR'
]

In [21]:
import duckdb
import pprint

# Connect to local DuckDB file
db_path = "C:\\Users\\Keerti A M\\Documents\\New folder\\data\\india_market (2).duckdb"
conn = duckdb.connect(db_path)

# Get all table names
tables_df = conn.execute("SHOW TABLES").fetchdf()
table_names = tables_df["name"].tolist()

all_column_names = {}

# Get column names for each table
for table_name in table_names:
    columns_desc = conn.execute(
        f"SELECT * FROM {table_name} LIMIT 0"
    ).description

    column_names = [desc[0] for desc in columns_desc]

    all_column_names[table_name] = column_names

# Print schema
pprint.pprint(all_column_names)

# Close connection


{'companies': ['symbol',
               'company_name',
               'sector',
               'industry',
               'isin',
               'listing_date',
               'cap_category',
               'avg_mcap_cr',
               'exchange',
               'bse_code'],
 'company_betas': ['symbol',
                   'regression_beta',
                   'blume_beta',
                   'damodaran_beta',
                   'blended_beta',
                   'r_squared',
                   'n_obs',
                   'computed_date'],
 'corporate_actions': ['symbol', 'date', 'action_type', 'value', 'remarks'],
 'fetched_dates': ['date'],
 'financials': ['symbol',
                'period',
                'year',
                'quarter',
                'revenue',
                'net_profit',
                'ebitda',
                'eps',
                'assets',
                'liabilities',
                'equity',
                'debt',
                'operating_cash_

In [22]:
import pandas as pd

# Fetch company details
companies_query = f"""
SELECT symbol, company_name, sector, industry, cap_category, avg_mcap_cr
FROM companies
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
"""
companies_df = conn.execute(companies_query).fetchdf()

# Fetch price data
prices_query = f"""
SELECT symbol, date, open, high, low, close, volume
FROM prices
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, date
"""
prices_df = conn.execute(prices_query).fetchdf()

# Convert date column to datetime objects
prices_df['date'] = pd.to_datetime(prices_df['date'])

# Merge company details with price data
combined_df = pd.merge(prices_df, companies_df, on='symbol', how='left')

print("Combined DataFrame after merging companies and prices:")
display(combined_df.head())

Combined DataFrame after merging companies and prices:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,cap_category,avg_mcap_cr
0,BAJFINANCE,2010-09-29,780.0,805.00,770.00,774.60,26000,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
1,BAJFINANCE,2010-09-30,775.0,783.45,766.25,773.55,17350,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
2,BAJFINANCE,2010-10-01,763.0,797.30,760.25,774.95,35941,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
3,BAJFINANCE,2010-10-04,788.0,795.00,775.00,783.40,31160,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856
4,BAJFINANCE,2010-10-05,775.1,788.00,772.05,779.00,32141,Bajaj Finance Limited,Finance,Finance,Large Cap,608063.504856


In [23]:
# Fetch financial data
financials_query = f"""
SELECT symbol, year, quarter, period, revenue, net_profit, ebitda, eps, assets, liabilities, equity, debt, operating_cash_flow, free_cash_flow, book_value_per_share, operating_profit, ebit, shares_outstanding, cash_equivalents
FROM financials
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, year, quarter
"""
financials_df = conn.execute(financials_query).fetchdf()

# Convert 'year' and 'quarter' into a single 'date' column
# Assuming fiscal year starts in April (common in India), so FY 2023 means April 2023 - March 2024
def create_financial_date(row):
    try:
        year = int(row['year'])
        period_type = row['period']

        if period_type == 'annual':
            # For annual reports, assume fiscal year end (March 31 of next calendar year)
            return pd.to_datetime(f"{year + 1}-03-31")
        elif period_type == 'Q1':
            # Q1 (April-June) of the fiscal year
            return pd.to_datetime(f"{year}-06-30")
        elif period_type == 'Q2':
            # Q2 (July-September) of the fiscal year
            return pd.to_datetime(f"{year}-09-30")
        elif period_type == 'Q3':
            # Q3 (October-December) of the fiscal year
            return pd.to_datetime(f"{year}-12-31")
        elif period_type == 'Q4':
            # Q4 (January-March) of the fiscal year, falls in the next calendar year
            return pd.to_datetime(f"{year + 1}-03-31")
        else:
            return pd.NaT
    except ValueError:
        return pd.NaT

financials_df['date'] = financials_df.apply(create_financial_date, axis=1)

# Drop the original 'year', 'quarter', and 'period' columns as they are no longer needed
financials_df = financials_df.drop(columns=['year', 'quarter', 'period'])

# Fetch shareholding data
shareholding_query = f"""
SELECT symbol, quarter_end, promoter_pct, fii_pct, dii_pct, public_pct
FROM shareholding
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
ORDER BY symbol, quarter_end
"""
shareholding_df = conn.execute(shareholding_query).fetchdf()
shareholding_df['quarter_end'] = pd.to_datetime(shareholding_df['quarter_end'])

# Fetch company betas
betas_query = f"""
SELECT symbol, blended_beta
FROM company_betas
WHERE symbol IN ({','.join([f"'{s}'" for s in symbols])})
"""
company_betas_df = conn.execute(betas_query).fetchdf()

# Ensure all date columns have the same precision (nanoseconds) for merge_asof
combined_df['date'] = combined_df['date'].astype('datetime64[ns]')
financials_df['date'] = financials_df['date'].astype('datetime64[ns]')
shareholding_df['quarter_end'] = shareholding_df['quarter_end'].astype('datetime64[ns]')

# Drop rows with NaT in the 'date' column of combined_df and financials_df, 
# and NaN in 'symbol' column to ensure clean keys for sorting and merging
combined_df = combined_df.dropna(subset=['date', 'symbol'])
financials_df = financials_df.dropna(subset=['date', 'symbol'])

# Ensure unique (symbol, date) pairs in combined_df and financials_df before sorting
# This is a defensive measure to prevent potential issues with merge_asof's strict sorting checks
combined_df = combined_df.drop_duplicates(subset=['symbol', 'date'])
financials_df = financials_df.drop_duplicates(subset=['symbol', 'date'])

# merge_asof requires the 'on' key to be sorted globally (not per-symbol).
# Sorting by ['symbol', 'date'] causes dates to reset for each symbol, breaking global order.
combined_df = combined_df.sort_values(by='date').reset_index(drop=True)
financials_df = financials_df.sort_values(by='date').reset_index(drop=True)
shareholding_df = shareholding_df.sort_values(by='quarter_end').reset_index(drop=True)

# Merge financials using merge_asof
combined_df = pd.merge_asof(combined_df, financials_df, on='date', by='symbol', direction='backward')

# Merge shareholding using merge_asof
# Rename 'quarter_end' to 'date' temporarily for merge_asof
shareholding_df_temp = shareholding_df.rename(columns={'quarter_end': 'date'})
combined_df = pd.merge_asof(combined_df, shareholding_df_temp, on='date', by='symbol', direction='backward')

# Merge company betas (these are usually static or updated infrequently, so a simple merge is fine)
combined_df = pd.merge(combined_df, company_betas_df, on='symbol', how='left')

print("Combined DataFrame after merging financials, shareholding, and betas:")
display(combined_df.head())

Combined DataFrame after merging financials, shareholding, and betas:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,book_value_per_share,operating_profit,ebit,shares_outstanding,cash_equivalents,promoter_pct,fii_pct,dii_pct,public_pct,blended_beta
0,SBIN,1996-04-01,230.00,243.0,230.00,241.75,3787050,State Bank of India,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.831
1,HDFCBANK,1996-04-01,32.45,33.0,32.40,32.90,96300,HDFC Bank Limited,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.791
2,RELIANCE,1996-04-01,207.75,209.5,206.00,208.80,8836850,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.978
3,RELIANCE,1996-04-02,209.25,212.0,207.25,209.35,13118200,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.978
4,SBIN,1996-04-02,245.05,253.0,238.65,249.40,5067850,State Bank of India,Banks,Banks,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.831


In [24]:
# Fetch macro data
macro_query = """
SELECT date, metric, value
FROM macro_data
ORDER BY date
"""
macro_df = conn.execute(macro_query).fetchdf()
macro_df['date'] = pd.to_datetime(macro_df['date']).astype('datetime64[ns]')

# Pivot macro_df to have metrics as columns
macro_pivot_df = macro_df.pivot_table(index='date', columns='metric', values='value').reset_index()
macro_pivot_df.columns.name = None
macro_pivot_df = macro_pivot_df.sort_values(by='date').reset_index(drop=True)

# Use merge_asof so each row gets the most recent macro reading (macro data is monthly/quarterly)
combined_df = combined_df.sort_values(by='date').reset_index(drop=True)
combined_df = pd.merge_asof(combined_df, macro_pivot_df, on='date', direction='backward')

print("Combined DataFrame after merging macro data:")
display(combined_df.head())

Combined DataFrame after merging macro data:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,us_cpi_index,us_cpi_yoy,us_fed_rate_pct,us_gdp_growth_pct,us_gdp_usd_bn,us_real_interest_rate,us_unemployment_pct,us_unemployment_pct_m,usd_inr,wti_crude_usd
0,SBIN,1996-04-01,230.00,243.0,230.00,241.75,3787050,State Bank of India,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
1,HDFCBANK,1996-04-01,32.45,33.0,32.40,32.90,96300,HDFC Bank Limited,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
2,RELIANCE,1996-04-01,207.75,209.5,206.00,208.80,8836850,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
3,RELIANCE,1996-04-02,209.25,212.0,207.25,209.35,13118200,Reliance Industries Limited,"Oil, Gas & Consumable Fuels",Oil,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN
4,SBIN,1996-04-02,245.05,253.0,238.65,249.40,5067850,State Bank of India,Banks,Banks,...,155.5,NaN,5.31,NaN,NaN,NaN,NaN,5.5,NaN,NaN


In [25]:
import numpy as np

financial_cols = ['revenue', 'net_profit', 'equity', 'debt', 'ebitda', 'operating_cash_flow', 'book_value_per_share']

# Forward-fill per symbol so each day carries the last known quarterly value
combined_df = combined_df.sort_values(by=['symbol', 'date'])
combined_df[financial_cols] = combined_df.groupby('symbol')[financial_cols].ffill()

# profit_margin = net_profit / revenue
combined_df['profit_margin'] = combined_df['net_profit'] / combined_df['revenue']

# roe = net_profit / equity
combined_df['roe'] = combined_df['net_profit'] / combined_df['equity']

# debt_to_equity = debt / equity
combined_df['debt_to_equity'] = combined_df['debt'] / combined_df['equity']

# ebitda_margin = ebitda / revenue
combined_df['ebitda_margin'] = combined_df['ebitda'] / combined_df['revenue']

# book_to_price = book_value_per_share / close
combined_df['book_to_price'] = combined_df['book_value_per_share'] / combined_df['close']

# cash_flow_margin = operating_cash_flow / revenue
combined_df['cash_flow_margin'] = combined_df['operating_cash_flow'] / combined_df['revenue']

# Replace any inf values from division by zero with NaN
combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Combined DataFrame after calculating derived financial features:")
display(combined_df.head())

Combined DataFrame after calculating derived financial features:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,us_unemployment_pct,us_unemployment_pct_m,usd_inr,wti_crude_usd,profit_margin,roe,debt_to_equity,ebitda_margin,book_to_price,cash_flow_margin
18920,BAJFINANCE,2010-09-29,780.0,805.00,770.00,774.60,26000,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,45.050,77.86,NaN,NaN,NaN,NaN,NaN,NaN
18933,BAJFINANCE,2010-09-30,775.0,783.45,766.25,773.55,17350,Bajaj Finance Limited,Finance,Finance,...,NaN,9.5,44.690,79.97,NaN,NaN,NaN,NaN,NaN,NaN
18940,BAJFINANCE,2010-10-01,763.0,797.30,760.25,774.95,35941,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.563,81.58,NaN,NaN,NaN,NaN,NaN,NaN
18951,BAJFINANCE,2010-10-04,788.0,795.00,775.00,783.40,31160,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.413,81.47,NaN,NaN,NaN,NaN,NaN,NaN
18954,BAJFINANCE,2010-10-05,775.1,788.00,772.05,779.00,32141,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,44.450,82.82,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# Ensure data is sorted by symbol and date for correct rolling calculations
combined_df = combined_df.sort_values(by=['symbol', 'date'])
combined_df['volume'] = combined_df['volume'].astype(float)

# Calculate daily_return
combined_df['daily_return'] = combined_df.groupby('symbol')['close'].pct_change()

# Calculate n-day returns
for days in [5, 20]:
    combined_df[f'{days}d_return'] = combined_df.groupby('symbol')['close'].pct_change(periods=days)

# Calculate Moving Averages (MA)
for days in [50, 200]:
    combined_df[f'{days}d_ma'] = combined_df.groupby('symbol')['close'].rolling(window=days).mean().reset_index(level=0, drop=True)

# Calculate 20d_volatility (rolling standard deviation of daily returns)
combined_df['20d_volatility'] = combined_df.groupby('symbol')['daily_return'].rolling(window=20).std().reset_index(level=0, drop=True)

# volume_ratio (e.g., Volume / 20-day Average Volume)
combined_df['20d_avg_volume'] = combined_df.groupby('symbol')['volume'].rolling(window=20).mean().reset_index(level=0, drop=True)
combined_df['volume_ratio'] = combined_df['volume'] / combined_df['20d_avg_volume']

# Display the final DataFrame with all features
print("Final DataFrame with all calculated features:")
display(combined_df.head())

Final DataFrame with all calculated features:


,symbol,date,open,high,low,close,volume,company_name,sector,industry,...,book_to_price,cash_flow_margin,daily_return,5d_return,20d_return,50d_ma,200d_ma,20d_volatility,20d_avg_volume,volume_ratio
18920,BAJFINANCE,2010-09-29,780.0,805.00,770.00,774.60,26000.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18933,BAJFINANCE,2010-09-30,775.0,783.45,766.25,773.55,17350.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,-0.001356,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18940,BAJFINANCE,2010-10-01,763.0,797.30,760.25,774.95,35941.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,0.001810,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18951,BAJFINANCE,2010-10-04,788.0,795.00,775.00,783.40,31160.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,0.010904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18954,BAJFINANCE,2010-10-05,775.1,788.00,772.05,779.00,32141.0,Bajaj Finance Limited,Finance,Finance,...,NaN,NaN,-0.005617,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# --- NaN Handling ---

combined_df = combined_df[combined_df['200d_ma'].notna()].reset_index(drop=True)


# Step 2: Forward-fill macro and financial ratio columns per symbol
macro_cols = [c for c in combined_df.columns if c.startswith('us_') or c in ['usd_inr', 'wti_crude_usd']]
financial_ratio_cols = ['profit_margin', 'roe', 'debt_to_equity', 'ebitda_margin', 'book_to_price', 'cash_flow_margin']
fill_cols = macro_cols + financial_ratio_cols
combined_df[fill_cols] = combined_df.groupby('symbol')[fill_cols].transform('ffill')

# Step 3: Drop any rows still NaN in critical columns
critical_cols = ['daily_return', '5d_return', '20d_return', '50d_ma', '200d_ma', '20d_volatility', 'volume_ratio']
combined_df = combined_df.dropna(subset=critical_cols).reset_index(drop=True)

print("Shape after NaN handling:", combined_df.shape)
print("\nRemaining NaN counts:")
nan_counts = combined_df.isna().sum()
print(nan_counts[nan_counts > 0])

Shape after NaN handling: (52150, 67)

Remaining NaN counts:
revenue                  28114
net_profit               28114
ebitda                   28114
eps                      28134
assets                   28114
liabilities              28114
equity                   28114
debt                     28114
operating_cash_flow      28114
free_cash_flow           28361
book_value_per_share     28134
operating_profit         28114
ebit                     38912
shares_outstanding       28134
cash_equivalents         29349
promoter_pct             45168
fii_pct                  45168
dii_pct                  45168
public_pct               45168
brent_crude_usd          12917
bse_sensex                 549
gold_inr                  8088
gold_usd                  4724
india_cpi_yoy            51968
india_gdp_growth_pct     51968
india_gdp_usd_bn         51968
india_repo_rate_pct      50502
india_vix                12596
india_wpi_proxy_pct      51968
nifty50                  11753
us_cpi_in

In [28]:
combined_df.to_parquet("C:\\Users\\Keerti A M\\Documents\\New folder\\data\\combined_features.parquet", index=False)
print("Saved successfully. Shape:", combined_df.shape)

Saved successfully. Shape: (52150, 67)
